# 프로젝트 Notebook 02. 공공데이터 API 수집

목표: HTTP 요청, 인증키, JSON 파싱, 페이지네이션, 오류 처리와 원본 보존을 학습한다.
API 키가 없어도 내장 예제 응답으로 파싱 부분을 실행할 수 있다.

In [ ]:
from pathlib import Path
import sys, subprocess

REPO_URL = "https://github.com/niko2204/bigdataservice.git"
if "google.colab" in sys.modules:
    ROOT = Path("/content/bigdataservice")
    if not ROOT.exists():
        subprocess.run(["git", "clone", "-q", REPO_URL, str(ROOT)], check=True)
else:
    candidates = [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]
    ROOT = next((p.resolve() for p in candidates if (p / "data").exists()), None)
    if ROOT is None:
        raise FileNotFoundError("bigdataservice 저장소 안에서 실행하세요.")
print("저장소:", ROOT)

## 1. 요청 구성과 키 관리

키는 .env 또는 Colab Secrets에 저장한다. Notebook에 직접 적거나 출력하지 않는다.

In [ ]:
import os, json, time
from datetime import datetime, timezone
import requests
import pandas as pd
from dotenv import load_dotenv

load_dotenv(ROOT / ".env")
API_KEY = os.getenv("DATA_GO_KR_API_KEY")
BASE_URL = "https://apis.data.go.kr/B553077/api/open/sdsc2/storeListInRadius"
params = {
    "serviceKey": API_KEY, "cx": 126.3922, "cy": 34.8118,
    "radius": 1000, "pageNo": 1, "numOfRows": 10, "type": "json",
}
print("인증키 설정 여부:", bool(API_KEY))

## 2. 키 없이 JSON 파싱 연습

중첩 구조의 키를 한 단계씩 확인한다.

In [ ]:
DEMO_RESPONSE = {
    "body": {
        "totalCount": 2,
        "items": [
            {"bizesId": "S001", "bizesNm": "유달커피", "indsSclsNm": "카페", "lon": 126.3821, "lat": 34.7915},
            {"bizesId": "S002", "bizesNm": "평화카페", "indsSclsNm": "카페", "lon": 126.4172, "lat": 34.8040},
        ],
    }
}
body = DEMO_RESPONSE["body"]
demo_df = pd.DataFrame(body.get("items", []))
display(demo_df)
assert len(demo_df) == body["totalCount"]

## 3. 실제 10건 호출

HTTP 200만 확인하지 말고 본문의 결과 코드와 건수도 확인한다.

In [ ]:
if API_KEY:
    response = requests.get(BASE_URL, params=params, timeout=20)
    response.raise_for_status()
    payload = response.json()
    print("상위 키:", payload.keys())
else:
    print("API 키가 없어 예제 응답을 사용합니다.")
    payload = DEMO_RESPONSE

## 4. 재사용 가능한 파서

예상 구조가 아니면 조용히 빈 표를 만들지 않고 오류를 발생시킨다.

In [ ]:
def parse_items(payload):
    body = payload.get("body")
    if not isinstance(body, dict):
        raise ValueError("응답에 body 객체가 없습니다.")
    items = body.get("items", [])
    if isinstance(items, dict):
        items = items.get("item", [])
    if not isinstance(items, list):
        raise ValueError("items가 리스트가 아닙니다.")
    return pd.DataFrame(items), int(body.get("totalCount", len(items)))

page_df, total_count = parse_items(payload)
print("현재:", len(page_df), "전체:", total_count)
display(page_df.head())

## 5. 페이지네이션

종료 조건과 최대 페이지를 함께 둔다. 페이지, 현재 건수, 누적 건수와 전체 건수를 로그로 남긴다.

In [ ]:
def collect_pages(base_url, base_params, max_pages=20, wait_seconds=.2):
    if not base_params.get("serviceKey"):
        raise RuntimeError("DATA_GO_KR_API_KEY가 필요합니다.")
    frames, collected = [], 0
    for page in range(1, max_pages + 1):
        query = {**base_params, "pageNo": page}
        response = requests.get(base_url, params=query, timeout=20)
        response.raise_for_status()
        frame, total = parse_items(response.json())
        print(f"page={page}, batch={len(frame)}, accumulated={collected}, total={total}")
        if frame.empty:
            break
        frames.append(frame); collected += len(frame)
        if collected >= total:
            break
        time.sleep(wait_seconds)
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()

## 6. 수집 메타데이터

In [ ]:
metadata = {
    "source": BASE_URL,
    "collected_at_utc": datetime.now(timezone.utc).isoformat(),
    "parameters_without_key": {k:v for k,v in params.items() if k != "serviceKey"},
    "rows_in_current_page": len(page_df),
}
print(json.dumps(metadata, ensure_ascii=False, indent=2))

## 7. 독립 연습

1. 실제 응답 필드 다섯 개의 정의를 활용가이드에서 찾아 표로 작성한다.
2. 100건 이상 수집하고 식별자 유일성, 좌표 범위와 결측을 검사한다.
3. timeout·429·500대 오류를 제한적으로 재시도하도록 개선한다.
4. 원본 건수와 정제 CSV 건수 차이를 자동 기록한다.